# Pipeline Inspection: Training Pipeline (DLR-TomoSAR)

This notebook inspects the **deep-learning training pipeline** orchestrated by
`pipelines.training_pipeline.pipeline.TrainingPipeline.run()` for tomographic SAR
reconstruction. The model ingests SAR image patches of shape `(B, in_channels, H, W)`
and predicts per-pixel Gaussian-mixture parameters of shape `(B, 3K, H, W)`, where `K`
is the number of Gaussian components. These parameters reconstruct the elevation /
height profile (the **tomogram** curve) along a discrete spectral `x_axis`, which is
compared against the experimental curve to compute the loss.

`TrainingPipeline.run()` sequences: dataset build → model + AdamW optimizer build →
warmup + LR scheduler setup → loss construction → optional loss-scale probe → the
training loop. Inside every optimizer step the trainer composes: forward + loss →
backward → gradient clipping → optimizer step → EMA update → per-epoch metric
aggregation → curriculum swap → checkpoint → LR scheduler step → early-stopping check.

This notebook verifies each of those sub-steps in isolation, with structural,
statistical, and semantic assertions, so behavioural equivalence can be demonstrated
before and after any refactor.

**Note on data availability.** The underlying SAR tomogram arrays and some external
repositories are not present in this environment. Each stage therefore exercises the
*real* pipeline component classes against config-derived synthetic tensors that match
the production shapes and ranges. The notebook is executable top-to-bottom once the
real data is mounted; substituting `TrainingPipeline.run()` for the synthetic batch is
then a one-line change documented in the final stage.

In [ ]:
from __future__ import annotations

import os
import sys
import copy
import math
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS",      "1")
os.environ.setdefault("MKL_NUM_THREADS",      "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import torch
import torch.nn as nn
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family":        "serif",
    "font.size":          11,
    "axes.labelsize":     12,
    "axes.titlesize":     13,
    "legend.fontsize":    10,
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
})

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pipelines").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIGURE_DIRECTORY = PROJECT_ROOT / "notebooks" / "figures" / "inspect_training_pipeline"
FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 0
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

from configuration.training_config                import (
    TrainerConfig, GaussianConfig, WarmupConfig, SchedulerConfig, EMAConfig,
    EarlyStoppingConfig, OptimizerConfig, TrainingConfigInner, OverfitConfig,
    LossConfig, LossCurriculumConfig, GradientClipperConfig, IOConfig,
)
from configuration.models_config                  import UNetConfig
from pipelines.training_pipeline.warmup           import Warmup
from pipelines.training_pipeline.scheduler        import Scheduler
from pipelines.training_pipeline.ema              import EMA
from pipelines.training_pipeline.early_stopping   import EarlyStopping
from pipelines.training_pipeline.gradient_clipper import GradientClipper
from pipelines.training_pipeline.loss             import Loss
from pipelines.training_pipeline.loss_components  import LossComponents
from pipelines.training_pipeline.train_step       import TrainStep
from pipelines.training_pipeline.metric_aggregator import MetricAggregator
from pipelines.training_pipeline.curriculum       import CurriculumController
from pipelines.training_pipeline.checkpoint       import Checkpoint
from pipelines.training_pipeline.overfit          import OverfitManager


class NotebookLogger:
    def section(self, *args, **kwargs):    self._emit("SECTION", args)
    def subsection(self, *args, **kwargs): self._emit("  -", args)
    def info(self, *args, **kwargs):       self._emit("INFO", args)
    def warning(self, *args, **kwargs):    self._emit("WARN", args)
    def kv_table(self, mapping, *args, **kwargs):
        for key, value in mapping.items():
            print(f"    {key:>20} : {value}")
    def metrics_table(self, rows, columns, title=None, *args, **kwargs):
        if title:
            print(f"    [{title}]")
        for row in rows:
            print("    " + "  ".join(f"{column}={row.get(column)}" for column in columns))
    def _emit(self, tag, args):
        print(f"[{tag}] " + " ".join(str(argument) for argument in args))


class NotebookTracker:
    def __init__(self, debug: bool = False):
        self.debug   = debug
        self.scalars = {}
    def log_scalar(self, name, value, step):
        self.scalars.setdefault(name, []).append((step, float(value)))
    def log_histogram(self, name, values, step): pass
    def log_metrics(self, prefix, mapping, step): pass


class IdentityNormalizer:
    def normalize_output(self, tensor):
        return tensor
    def denormalize_output(self, tensor):
        return tensor


notebook_logger    = NotebookLogger()
notebook_tracker   = NotebookTracker(debug=False)
identity_normalizer = IdentityNormalizer()

GAUSSIAN_CONFIG = GaussianConfig(n_default_gaussians=3, x_min=-30.0, x_max=30.0, amp_max=1000.0, params_per_gaussian=3)

TRAINER_CONFIG = TrainerConfig(
    gaussian         = GAUSSIAN_CONFIG,
    warmup           = WarmupConfig(),
    scheduler        = SchedulerConfig(),
    ema              = EMAConfig(),
    early_stopping   = EarlyStoppingConfig(),
    optimizer        = OptimizerConfig(),
    training         = TrainingConfigInner(),
    overfit          = OverfitConfig(),
    curriculum       = LossCurriculumConfig(),
    gradient_clipper = GradientClipperConfig(),
)

NUMBER_OF_GAUSSIANS       = GAUSSIAN_CONFIG.n_default_gaussians
PARAMS_PER_GAUSSIAN       = GAUSSIAN_CONFIG.params_per_gaussian
OUTPUT_CHANNELS           = PARAMS_PER_GAUSSIAN * NUMBER_OF_GAUSSIANS
INPUT_CHANNELS            = 1
PATCH_HEIGHT              = 16
PATCH_WIDTH               = 16
BATCH_SIZE                = 4
SPECTRAL_SAMPLE_POINTS    = 64

SPECTRAL_X_AXIS = np.linspace(GAUSSIAN_CONFIG.x_min, GAUSSIAN_CONFIG.x_max, SPECTRAL_SAMPLE_POINTS, dtype=np.float32)

COMPUTE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root            : {PROJECT_ROOT}")
print(f"Figure directory        : {FIGURE_DIRECTORY}")
print(f"Compute device          : {COMPUTE_DEVICE}")
print(f"Gaussian components K    : {NUMBER_OF_GAUSSIANS}")
print(f"Output channels (3K)     : {OUTPUT_CHANNELS}")
print(f"Spectral sample points N : {SPECTRAL_SAMPLE_POINTS}")
print(f"Synthetic patch (H, W)   : ({PATCH_HEIGHT}, {PATCH_WIDTH})")

## Pipeline Overview

```
SAR patches  (B, in_channels, H, W)
  └─► Stage 1:  Run metadata + seeding        →  run directory, fixed RNG, config snapshot
        └─► Stage 2:  Model + AdamW build      →  nn.Module (B,3K,H,W head) + param-group optimizer
              └─► Stage 3:  Warmup setup        →  per-step LR factor ramp α(s) ∈ [α_start, 1]
                    └─► Stage 4:  LR scheduler   →  per-epoch LR factor (cosine annealing) × warmup
                          └─► Stage 5:  Loss      →  weighted sum of curve + parameter terms → scalar
                                └─► Stage 6:  Train step  →  forward + backward + opt step (1 batch)
                                      └─► Stage 7:  Gradient clipper  →  ||g||₂ rescaled to threshold
                                            └─► Stage 8:  EMA update    →  shadow θ̃ = γθ̃ + (1-γ)θ
                                                  └─► Stage 9:  Metric aggregator  →  epoch-mean components
                                                        └─► Stage 10: Curriculum swap  →  loss cfg replace
                                                              └─► Stage 11: Early stopping  →  stop flag
                                                                    └─► Stage 12: Checkpoint  →  best_model.pt
                                                                          └─► Stage 13: Overfit probe → loss→0
```

| Stage | Callable | Input | Output |
|-------|----------|-------|--------|
| 1  | `TrainingRunMetadata` / `torch.manual_seed` | `TrainerConfig`, model name, seed | Run directory tree, deterministic RNG, serialized config |
| 2  | `TrainingPipeline._build_model` + `Trainer._build_optimizer` | `in_channels`, `out_channels` | `nn.Module` with `(B,3K,H,W)` head + AdamW param groups |
| 3  | `Warmup.factor()` / `Warmup.step()` | global step counter | LR scaling factor `α(s) ∈ [α_start, 1]` |
| 4  | `Scheduler.step(epoch, metric)` | epoch index, validation metric | per-epoch LR list (annealed × warmup) |
| 5  | `Loss.__call__(pred, gt)` | `(B,3K,H,W)` pred + gt params | dict with `total_loss`, `components`, `weighted` |
| 6  | `TrainStep.step(...)` | one batch of images + gt params | `(loss, loss_dict)`, optimizer advanced |
| 7  | `GradientClipper.maybe_clip(model, step)` | model with populated `.grad` | clipped global grad norm |
| 8  | `EMA.update / apply_to / restore` | model parameters | shadow weights, swap-in for eval |
| 9  | `MetricAggregator.add / reduce_*` | per-step `loss_dict` | epoch-mean component / weighted dicts |
| 10 | `CurriculumController.maybe_swap(epoch)` | epoch index | swapped loss config (warmup → complete) |
| 11 | `EarlyStopping.__call__(val_loss, model, epoch)` | validation loss | stop flag, best-weight snapshot |
| 12 | `Checkpoint.step(val_loss, epoch, trainer)` | validation loss, trainer state | `best_model.pt` on improvement |
| 13 | `OverfitManager.setup_loaders / check_stop` | train/val/test loaders | single-batch repeated loader, loss → 0 |

---
## Stage 1: Run Metadata and Deterministic Seeding

**Callable:** `TrainingPipeline.__init__()` → `torch.manual_seed(seed)` + `TrainingRunMetadata`
**Input:** `TrainerConfig`, model name, integer `seed`
**Output:** a run-directory tree (`tensorboard/`, `docs/`, `logs/`, `meta/`, `checkpoints/`), a deterministic RNG state, and the serialized `TrainerConfig`.

The pipeline first fixes the random state so the whole run is reproducible:

$$
\text{torch.manual\_seed}(s), \qquad \text{cudnn.deterministic} \leftarrow \text{True}
$$

It then resolves a run directory `base_logdir / run_<model>_<timestamp>` and snapshots the config to JSON.

> **What you should see:** Two independently seeded draws with the *same* seed produce
> bit-identical tensors (max abs difference exactly `0.0`). The config snapshot must
> round-trip the key fields used downstream: `gaussian.n_default_gaussians = 3`,
> `gaussian.params_per_gaussian = 3`, so `out_channels = 9`. The `x_axis` must be a
> strictly increasing length-`N` vector spanning `[x_min, x_max] = [-30, 30]`.

In [ ]:
def reproducible_draw(seed: int) -> torch.Tensor:
    torch.manual_seed(seed)
    return torch.randn(BATCH_SIZE, INPUT_CHANNELS, PATCH_HEIGHT, PATCH_WIDTH)

first_seeded_draw  = reproducible_draw(RANDOM_SEED)
second_seeded_draw = reproducible_draw(RANDOM_SEED)

resolved_output_channels = TRAINER_CONFIG.gaussian.params_per_gaussian * TRAINER_CONFIG.gaussian.n_default_gaussians
config_snapshot          = {
    "n_default_gaussians" : TRAINER_CONFIG.gaussian.n_default_gaussians,
    "params_per_gaussian" : TRAINER_CONFIG.gaussian.params_per_gaussian,
    "out_channels"        : resolved_output_channels,
    "x_min"               : TRAINER_CONFIG.gaussian.x_min,
    "x_max"               : TRAINER_CONFIG.gaussian.x_max,
}

In [ ]:
print("First  seeded draw  shape :", tuple(first_seeded_draw.shape),  "dtype:", first_seeded_draw.dtype)
print("Second seeded draw  shape :", tuple(second_seeded_draw.shape), "dtype:", second_seeded_draw.dtype)
print("Max abs difference        :", float((first_seeded_draw - second_seeded_draw).abs().max()))
print()
for key, value in config_snapshot.items():
    print(f"  config_snapshot[{key!r:>22}] = {value}")
print()
print("x_axis length             :", SPECTRAL_X_AXIS.shape[0])
print("x_axis span               :", (float(SPECTRAL_X_AXIS.min()), float(SPECTRAL_X_AXIS.max())))
print("x_axis dtype              :", SPECTRAL_X_AXIS.dtype)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 3.6))

axes[0].plot(np.arange(SPECTRAL_X_AXIS.shape[0]), SPECTRAL_X_AXIS, color="#1f4e79", linewidth=1.6)
axes[0].set_title("Stage 1 — Run Metadata: spectral sampling axis")
axes[0].set_xlabel("sample index n")
axes[0].set_ylabel("elevation / height (m)")

axes[1].hist(first_seeded_draw.flatten().numpy(), bins=40, color="#8c6d31", edgecolor="white", linewidth=0.4)
axes[1].set_title("Stage 1 — Run Metadata: seeded input draw")
axes[1].set_xlabel("normalized SAR amplitude (value)")
axes[1].set_ylabel("count")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_01_metadata.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_01_metadata.pdf")

In [ ]:
spacing_is_increasing = bool(np.all(np.diff(SPECTRAL_X_AXIS) > 0))

checks = [
    ("Same seed yields identical draws",        float((first_seeded_draw - second_seeded_draw).abs().max()) == 0.0),
    ("out_channels equals 3K from config",      resolved_output_channels == OUTPUT_CHANNELS),
    ("params_per_gaussian matches config",      TRAINER_CONFIG.gaussian.params_per_gaussian == PARAMS_PER_GAUSSIAN),
    ("x_axis length equals N sample points",    SPECTRAL_X_AXIS.shape[0] == SPECTRAL_SAMPLE_POINTS),
    ("x_axis strictly increasing",              spacing_is_increasing),
    ("x_axis spans configured height range",    math.isclose(float(SPECTRAL_X_AXIS.min()), TRAINER_CONFIG.gaussian.x_min) and math.isclose(float(SPECTRAL_X_AXIS.max()), TRAINER_CONFIG.gaussian.x_max)),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 1: Run Metadata and Seeding

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Two runs with same seed diverge | `torch.manual_seed` called after model/optimizer construction, or `cudnn.deterministic` left False | Re-seed and diff two draws; confirm seeding precedes any `randn` / weight init |
| `out_channels` does not equal `3K` | `n_default_gaussians` set on the wrong config object (dataset vs trainer gaussian) | Print `params_per_gaussian * n_default_gaussians` and compare to model head |
| `x_axis` reversed or off-range | `x_min`/`x_max` swapped, or `np.linspace` length taken from wrong tomogram axis | Assert `np.diff(x_axis) > 0` and endpoints equal `(x_min, x_max)` |
| Config snapshot missing fields | `to_dict()` fallback used instead of `asdict`, dropping nested dataclasses | Inspect the serialized JSON keys against `TrainerConfig` fields |

**Passing to Stage 2:** `resolved_output_channels` — `int` (= 9), the head width `3K` the model must emit; and `SPECTRAL_X_AXIS` — `float32[N]`, the reconstruction grid.

---
## Stage 2: Model and AdamW Optimizer Build

**Callable:** `TrainingPipeline._build_model(in_channels, out_channels)` → `models.get_model` ; `Trainer._build_optimizer(param_groups)` → `torch.optim.AdamW`
**Input:** `in_channels` (SAR patch channels), `out_channels = 3K` (Gaussian params)
**Output:** an `nn.Module` mapping `(B, in_channels, H, W) → (B, 3K, H, W)`, and an `AdamW` optimizer over per-block parameter groups (encoder / bottleneck / decoder / output_head), each with its own LR and weight decay.

The AdamW update for a parameter $\theta$ with LR $\eta$, decay $\lambda$ is:

$$
\theta_{t+1} = \theta_t - \eta\!\left( \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon} + \lambda \theta_t \right)
$$

> **What you should see:** Forward output shape exactly `(B, 3K, H, W)` =
> `(4, 9, 16, 16)`, finite everywhere. The parameter count is positive and matches the
> sum over `requires_grad` tensors. The optimizer holds one param group per non-empty
> block; the base LRs read back equal the per-block LRs in `UNetConfig`
> (`encoder_lr`, `bottleneck_lr`, `decoder_lr`, `output_head_lr`).

In [ ]:
from models import get_model

reconstruction_model, model_configuration = get_model(
    "unet",
    in_channels  = INPUT_CHANNELS,
    out_channels = OUTPUT_CHANNELS,
    features     = [16, 32, 64],
)
reconstruction_model = reconstruction_model.to(COMPUTE_DEVICE)

parameter_groups = model_configuration.get_param_groups(reconstruction_model)
base_learning_rates = [float(group["lr"]) for group in parameter_groups]

for group in parameter_groups:
    group.setdefault("betas",        tuple(TRAINER_CONFIG.optimizer.betas))
    group.setdefault("eps",          TRAINER_CONFIG.optimizer.eps)
    group.setdefault("weight_decay", TRAINER_CONFIG.optimizer.weight_decay)

adamw_optimizer = torch.optim.AdamW(parameter_groups)

synthetic_input_batch = torch.randn(BATCH_SIZE, INPUT_CHANNELS, PATCH_HEIGHT, PATCH_WIDTH, device=COMPUTE_DEVICE)
reconstruction_model.eval()
with torch.no_grad():
    model_forward_output = reconstruction_model(synthetic_input_batch)
reconstruction_model.train()

In [ ]:
trainable_parameter_count = sum(parameter.numel() for parameter in reconstruction_model.parameters() if parameter.requires_grad)

print("Model class               :", type(reconstruction_model).__name__)
print("Forward output shape      :", tuple(model_forward_output.shape))
print("Forward output dtype      :", model_forward_output.dtype)
print("Forward output finite     :", bool(torch.isfinite(model_forward_output).all()))
print("Trainable parameters      :", f"{trainable_parameter_count:,}")
print()
print("Optimizer param groups    :", len(adamw_optimizer.param_groups))
for group in adamw_optimizer.param_groups:
    tensor_count = sum(tensor.numel() for tensor in group["params"])
    print(f"  group {group.get('name', '?'):>12} : lr={group['lr']:.2e}  wd={group['weight_decay']:.2e}  params={tensor_count:,}")

In [ ]:
group_names         = [group.get("name", str(index)) for index, group in enumerate(adamw_optimizer.param_groups)]
group_learning_rate = [group["lr"] for group in adamw_optimizer.param_groups]
group_parameters    = [sum(tensor.numel() for tensor in group["params"]) for group in adamw_optimizer.param_groups]

figure, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].bar(range(len(group_names)), group_learning_rate, color="#1f4e79", edgecolor="white")
axes[0].set_xticks(range(len(group_names)))
axes[0].set_xticklabels(group_names, rotation=30, ha="right")
axes[0].set_title("Stage 2 — Model Build: per-group learning rate")
axes[0].set_xlabel("parameter group")
axes[0].set_ylabel("learning rate (unitless)")

axes[1].bar(range(len(group_names)), group_parameters, color="#8c6d31", edgecolor="white")
axes[1].set_xticks(range(len(group_names)))
axes[1].set_xticklabels(group_names, rotation=30, ha="right")
axes[1].set_title("Stage 2 — Model Build: parameters per group")
axes[1].set_xlabel("parameter group")
axes[1].set_ylabel("trainable parameters (count)")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_02_model_optimizer.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_02_model_optimizer.pdf")

In [ ]:
expected_output_shape = (BATCH_SIZE, OUTPUT_CHANNELS, PATCH_HEIGHT, PATCH_WIDTH)

checks = [
    ("Model is an nn.Module",                    isinstance(reconstruction_model, nn.Module)),
    ("Forward output shape is (B, 3K, H, W)",    tuple(model_forward_output.shape) == expected_output_shape),
    ("Forward output all finite",                bool(torch.isfinite(model_forward_output).all())),
    ("Trainable parameter count positive",       trainable_parameter_count > 0),
    ("Optimizer has at least one param group",   len(adamw_optimizer.param_groups) >= 1),
    ("All group LRs match config base LRs",      all(math.isclose(g["lr"], b) for g, b in zip(adamw_optimizer.param_groups, base_learning_rates))),
    ("AdamW betas propagated from config",       all(tuple(g["betas"]) == tuple(TRAINER_CONFIG.optimizer.betas) for g in adamw_optimizer.param_groups)),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 2: Model and Optimizer Build

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Output channels ≠ `3K` | `out_channels` override not passed to `get_model`, or noise head silently added | Print head out-channels vs `params_per_gaussian * n_gaussians` |
| All groups share one LR | param groups collapsed (e.g. `model.parameters()` passed instead of `get_param_groups`) | Count `optimizer.param_groups`; confirm distinct `name` keys |
| `betas` / `weight_decay` default to PyTorch values | `setdefault` skipped, so config optimizer settings never applied | Read back `group['betas']`, `group['weight_decay']` vs config |
| Empty param group raises in AdamW | a block (e.g. bottleneck) has zero params but is still appended | `get_param_groups` filters `len(params) > 0`; verify the filter ran |
| NaNs at first forward | weight init mode mismatched with activation (kaiming-relu on gelu) | Check `init_mode` vs `activation`; inspect `isfinite` on output |

**Passing to Stage 3:** `reconstruction_model` — `nn.Module` `(B,3K,H,W)`; `adamw_optimizer` — `AdamW` with per-block groups; `base_learning_rates` — `list[float]` consumed by the scheduler.

---
## Stage 3: Learning-Rate Warmup Setup

**Callable:** `Warmup.factor()` (read current factor) and `Warmup.step()` (advance one global step)
**Input:** the global optimizer-step counter
**Output:** a scalar LR scaling factor `α(s) ∈ [α_start, 1]`, multiplied onto every group LR until `warmup_steps` is reached.

For linear warmup at step $s \in [1, S_{\text{warmup}}]$ (the default mode):

$$
\alpha(s) = \alpha_{\text{start}} + \left(1 - \alpha_{\text{start}}\right)\frac{s}{S_{\text{warmup}}}, \qquad \alpha(s > S_{\text{warmup}}) = 1
$$

> **What you should see:** A monotonically non-decreasing ramp from
> `warmup_start_factor` (= 0.1) at step 1 toward `1.0` at step `warmup_steps` (= 200),
> then flat at `1.0`. `Warmup.step()` is invoked **once per training batch** (per global
> step) inside `train_epoch`, *not* per epoch — the ramp length is counted in steps.
> `is_finished()` flips to `True` exactly at `warmup_steps`.

In [ ]:
warmup_controller = Warmup(TRAINER_CONFIG, notebook_logger, notebook_tracker)

warmup_step_count   = TRAINER_CONFIG.warmup.warmup_steps
total_probe_steps   = int(warmup_step_count * 1.5)
warmup_factor_trace = []

for _ in range(total_probe_steps):
    warmup_factor_trace.append(warmup_controller.factor())
    warmup_controller.step()

warmup_factor_trace = np.asarray(warmup_factor_trace, dtype=np.float64)
warmup_controller_after = warmup_controller

In [ ]:
print("Warmup enabled            :", warmup_controller_after.enabled)
print("Warmup mode               :", warmup_controller_after.mode)
print("Configured warmup steps   :", warmup_step_count)
print("Start factor              :", TRAINER_CONFIG.warmup.warmup_start_factor)
print("Factor at step 1          :", float(warmup_factor_trace[0]))
print("Factor at warmup_steps-1  :", float(warmup_factor_trace[warmup_step_count - 1]))
print("Factor after completion   :", float(warmup_factor_trace[-1]))
print("Warmup is_finished        :", warmup_controller_after.is_finished())
print("Trace is non-decreasing   :", bool(np.all(np.diff(warmup_factor_trace[:warmup_step_count]) >= -1e-9)))

In [ ]:
def warmup_factor_trace_for_mode(mode_name: str) -> np.ndarray:
    config_copy = copy.deepcopy(TRAINER_CONFIG)
    config_copy.warmup.warmup_mode = mode_name
    controller  = Warmup(config_copy, NotebookLogger(), NotebookTracker())
    trace       = []
    for _ in range(total_probe_steps):
        trace.append(controller.factor())
        controller.step()
    return np.asarray(trace, dtype=np.float64)

step_axis = np.arange(total_probe_steps)

figure, axis = plt.subplots(figsize=(8, 4.2))
for mode_name, color in [("linear", "#1f4e79"), ("cosine", "#8c6d31"), ("polynomial", "#2e7d32"), ("exponential", "#7b1fa2")]:
    axis.plot(step_axis, warmup_factor_trace_for_mode(mode_name), label=mode_name, linewidth=1.6, color=color)

axis.axvline(warmup_step_count, color="0.4", linestyle="--", linewidth=1.0)
axis.text(warmup_step_count, 0.2, " warmup_steps", color="0.3", fontsize=9)
axis.set_title("Stage 3 — Warmup: LR scaling factor ramp by mode")
axis.set_xlabel("global optimizer step s")
axis.set_ylabel("LR scaling factor α(s) (unitless)")
axis.legend(title="warmup_mode", loc="lower right")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_03_warmup.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_03_warmup.pdf")

In [ ]:
linear_within_unit = bool(np.all(warmup_factor_trace >= -1e-9) and np.all(warmup_factor_trace <= 1.0 + 1e-9))

checks = [
    ("Initial factor equals start factor",      math.isclose(float(warmup_factor_trace[0]), TRAINER_CONFIG.warmup.warmup_start_factor, rel_tol=1e-6)),
    ("Ramp non-decreasing over warmup window",  bool(np.all(np.diff(warmup_factor_trace[:warmup_step_count]) >= -1e-9))),
    ("Factor reaches 1.0 after warmup",         math.isclose(float(warmup_factor_trace[-1]), 1.0, rel_tol=1e-9)),
    ("Factor never exceeds 1.0 or below 0",     linear_within_unit),
    ("is_finished True past warmup_steps",      warmup_controller_after.is_finished()),
    ("Counter advances once per step()",        warmup_controller_after.current_step == total_probe_steps if warmup_step_count <= 0 else warmup_controller_after.warmup_finished),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 3: Warmup

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| LR ramps over epochs instead of steps | `Warmup.step()` called once per epoch rather than per batch | Confirm `step()` is inside the batch loop in `train_epoch`, advancing `current_step` per global step |
| Warmup finishes far too early / late | `warmup_steps` counted in epochs, or dataloader length changed | Print `warmup_steps` against batches-per-epoch × intended epochs |
| Factor starts at 1.0 (no warmup effect) | `warmup_enabled` False or `warmup_start_factor` = 1.0 | Read `enabled` and `factor()` at step 1 |
| Factor exceeds 1.0 | wrong mode formula or `start_factor > 1` | Assert `0 ≤ α(s) ≤ 1` across the trace |
| Resumed run restarts warmup | `warmup_state` not restored from checkpoint | Compare `state_dict()['current_step']` before save and after load |

**Passing to Stage 4:** `warmup_controller` — `Warmup`, multiplies its `factor()` onto the scheduler output while not finished.

---
## Stage 4: Learning-Rate Scheduler Setup

**Callable:** `Scheduler.step(epoch, metric)`
**Input:** epoch index and (optionally) the validation metric
**Output:** a list of per-group learning rates: the base LR scaled by the schedule factor, then by the warmup factor while warmup is unfinished.

For cosine annealing (the default), with progress $p = \min(1, t / T_{\max})$:

$$
\eta_t = \eta_{\min} + \tfrac{1}{2}\left(\eta_{\max} - \eta_{\min}\right)\left(1 + \cos(\pi p)\right)
$$

> **What you should see:** With warmup already complete, the LR follows a smooth cosine
> decay from `base_lr` at epoch 0 to approximately `eta_min` at `T_max` epochs. The
> scheduler steps **per epoch** (`train()` calls `lr_scheduler.step(epoch, ...)` once
> per epoch), distinct from the per-step warmup. The returned list has one entry per
> parameter group, all sharing the same factor.

In [ ]:
finished_warmup = Warmup(TRAINER_CONFIG, NotebookLogger(), NotebookTracker())
while not finished_warmup.is_finished():
    finished_warmup.step()

learning_rate_scheduler = Scheduler(base_learning_rates, finished_warmup, TRAINER_CONFIG, notebook_logger, notebook_tracker)

scheduled_epochs        = TRAINER_CONFIG.scheduler.epochs
learning_rate_trace     = np.asarray([learning_rate_scheduler.step(epoch)[0] for epoch in range(scheduled_epochs)], dtype=np.float64)

In [ ]:
print("Scheduler type            :", learning_rate_scheduler.scheduler_type)
print("T_max (scheduled epochs)  :", scheduled_epochs)
print("eta_min                   :", TRAINER_CONFIG.scheduler.eta_min)
print("Base LR (group 0)         :", base_learning_rates[0])
print("LR at epoch 0             :", float(learning_rate_trace[0]))
print("LR at final epoch         :", float(learning_rate_trace[-1]))
print("LR monotonically decreasing (cosine):", bool(np.all(np.diff(learning_rate_trace) <= 1e-12)))
print("Returned LR list length   :", len(learning_rate_scheduler.step(0)))

In [ ]:
warmup_for_combined = Warmup(TRAINER_CONFIG, NotebookLogger(), NotebookTracker())
scheduler_for_combined = Scheduler(base_learning_rates, warmup_for_combined, TRAINER_CONFIG, NotebookLogger(), NotebookTracker())

batches_per_epoch = max(1, TRAINER_CONFIG.warmup.warmup_steps // 8)
combined_steps    = []
combined_values   = []
running_step      = 0
for epoch in range(scheduled_epochs):
    epoch_learning_rate = scheduler_for_combined.step(epoch)[0]
    for _ in range(batches_per_epoch):
        combined_steps.append(running_step)
        combined_values.append(epoch_learning_rate * (warmup_for_combined.factor() if not warmup_for_combined.is_finished() else 1.0))
        warmup_for_combined.step()
        running_step += 1

figure, axes = plt.subplots(1, 2, figsize=(11, 4.0))

axes[0].plot(np.arange(scheduled_epochs), learning_rate_trace, color="#1f4e79", linewidth=1.8)
axes[0].axhline(TRAINER_CONFIG.scheduler.eta_min, color="0.4", linestyle="--", linewidth=1.0)
axes[0].set_title("Stage 4 — Scheduler: cosine annealing (warmup complete)")
axes[0].set_xlabel("epoch t")
axes[0].set_ylabel("learning rate (unitless)")

axes[1].plot(combined_steps, combined_values, color="#8c6d31", linewidth=1.2)
axes[1].set_title("Stage 4 — Scheduler: warmup ramp × schedule decay")
axes[1].set_xlabel("global optimizer step")
axes[1].set_ylabel("effective learning rate (unitless)")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_04_scheduler.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_04_scheduler.pdf")

In [ ]:
returned_list_length = len(learning_rate_scheduler.step(0))

checks = [
    ("LR at epoch 0 equals base LR",            math.isclose(float(learning_rate_trace[0]), base_learning_rates[0], rel_tol=1e-6)),
    ("Cosine schedule non-increasing",          bool(np.all(np.diff(learning_rate_trace) <= 1e-12))),
    ("Final LR approaches eta_min",             float(learning_rate_trace[-1]) <= base_learning_rates[0] and float(learning_rate_trace[-1]) >= TRAINER_CONFIG.scheduler.eta_min - 1e-9),
    ("LR list has one entry per group",         returned_list_length == len(adamw_optimizer.param_groups)),
    ("All LRs strictly positive",               bool(np.all(learning_rate_trace > 0.0))),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 4: Scheduler

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| LR collapses to eta_min in a few epochs | `T_max` set to steps not epochs, or scheduler stepped per-batch | Confirm `scheduler.step` is called once per epoch in `train()` |
| LR never decays | scheduler type `constant`, or warmup factor masking the schedule | Print `scheduler_type` and the unmasked `step(epoch)` values |
| Double-counted warmup | warmup applied both in scheduler and again in optimizer update | Trace effective LR = schedule × warmup once only |
| Plateau scheduler never reduces | `metric=None` passed (eval skipped that epoch) so plateau counter never advances | Check `validation_frequency` vs scheduler patience |
| LR jumps after curriculum swap | `reset_lr` re-bases with `epoch_offset`; expected, not a bug | Inspect `_epoch_offset` after `Scheduler.reset` |

**Passing to Stage 5:** `learning_rate_scheduler` — `Scheduler`, feeds `_update_optimizer` once per epoch.

---
## Stage 5: Loss Function Construction

**Callable:** `Loss.__call__(pred_params, gt_params)`
**Input:** predicted and ground-truth Gaussian-parameter tensors `(B, 3K, H, W)` (normalized space)
**Output:** a dict `{ "total_loss": scalar, "components": {...}, "weighted": {...} }`. The loss reconstructs curves from both predictions and targets and compares them, optionally adding parameter-space terms.

The reconstructed curve for pixel $(b,h,w)$ is the Gaussian superposition

$$
\hat y(x_n) = \sum_{k=1}^{K} a_k \exp\!\left(-\frac{(x_n - \mu_k)^2}{2\sigma_k^2}\right)
$$

and the total loss is the weight-normalized sum of active terms

$$
\mathcal{L}_{\text{total}} = \frac{1}{\sum_i w_i^{\text{eff}}} \sum_{i \in \text{active}} w_i^{\text{eff}} \, \ell_i, \qquad w_i^{\text{eff}} = \alpha_i \cdot \text{norm}_i
$$

> **What you should see:** With MSE-curve and param-L1 enabled, `total_loss` is a finite
> non-negative scalar. `components` holds the raw per-term values; `weighted` holds each
> times its effective weight `α·norm`. When predictions equal targets, every curve term
> and the total collapse to ~0. The reconstructed curves have shape `(B, N, H, W)`.

In [ ]:
inspection_loss_config = LossConfig(
    use_mse_curve    = True,
    weight_mse_curve = 1.0,
    use_param_l1     = True,
    weight_param_l1  = 0.1,
    param_match      = "sort_gt_by_mu",
)

spectral_axis_tensor = torch.tensor(SPECTRAL_X_AXIS, device=COMPUTE_DEVICE, dtype=torch.float32)

loss_function = Loss(
    x_axis       = spectral_axis_tensor,
    logger       = notebook_logger,
    tracker      = notebook_tracker,
    gaussian_cfg = GAUSSIAN_CONFIG,
    loss_cfg     = inspection_loss_config,
    norm_stats   = identity_normalizer,
)

def make_physical_gaussian_parameters() -> torch.Tensor:
    amplitudes = torch.rand(BATCH_SIZE, NUMBER_OF_GAUSSIANS, 1, PATCH_HEIGHT, PATCH_WIDTH, device=COMPUTE_DEVICE) * 5.0 + 1.0
    means      = torch.rand(BATCH_SIZE, NUMBER_OF_GAUSSIANS, 1, PATCH_HEIGHT, PATCH_WIDTH, device=COMPUTE_DEVICE) * 40.0 - 20.0
    sigmas     = torch.rand(BATCH_SIZE, NUMBER_OF_GAUSSIANS, 1, PATCH_HEIGHT, PATCH_WIDTH, device=COMPUTE_DEVICE) * 3.0 + 1.0
    stacked    = torch.cat([amplitudes, means, sigmas], dim=2)
    return stacked.reshape(BATCH_SIZE, OUTPUT_CHANNELS, PATCH_HEIGHT, PATCH_WIDTH)

ground_truth_parameters    = make_physical_gaussian_parameters()
predicted_parameters       = ground_truth_parameters + 0.3 * torch.randn_like(ground_truth_parameters)

loss_output_dictionary     = loss_function(predicted_parameters, ground_truth_parameters)
loss_output_identical      = loss_function(ground_truth_parameters.clone(), ground_truth_parameters.clone())

reconstructed_curves       = loss_function.reconstruct(ground_truth_parameters.float())

In [ ]:
print("total_loss                :", float(loss_output_dictionary["total_loss"]))
print("total_loss (pred == gt)   :", float(loss_output_identical["total_loss"]))
print("reconstructed curve shape :", tuple(reconstructed_curves.shape))
print()
print("Raw components:")
for name, value in loss_output_dictionary["components"].items():
    print(f"    {name:>22} = {float(value): .6e}")
print("Weighted contributions:")
for name, value in loss_output_dictionary["weighted"].items():
    print(f"    {name:>22} = {float(value): .6e}")

In [ ]:
component_names  = list(loss_output_dictionary["components"].keys())
component_values = [float(loss_output_dictionary["components"][name]) for name in component_names]
weighted_values  = [float(loss_output_dictionary["weighted"].get(name, 0.0)) for name in component_names]

figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))

bar_positions = np.arange(len(component_names))
bar_width     = 0.4
axes[0].bar(bar_positions - bar_width / 2, component_values, bar_width, label="raw component", color="#1f4e79", edgecolor="white")
axes[0].bar(bar_positions + bar_width / 2, weighted_values,  bar_width, label="weighted",      color="#8c6d31", edgecolor="white")
axes[0].set_xticks(bar_positions)
axes[0].set_xticklabels(component_names, rotation=30, ha="right")
axes[0].set_title("Stage 5 — Loss: raw vs effective-weighted components")
axes[0].set_xlabel("loss term")
axes[0].set_ylabel("loss value (unitless)")
axes[0].legend(loc="upper right")

sample_curve_predicted = loss_function.reconstruct(predicted_parameters.float())[0, :, 0, 0].detach().cpu().numpy()
sample_curve_target    = reconstructed_curves[0, :, 0, 0].detach().cpu().numpy()
axes[1].plot(SPECTRAL_X_AXIS, sample_curve_target,    label="target curve",    color="#1f4e79", linewidth=1.8)
axes[1].plot(SPECTRAL_X_AXIS, sample_curve_predicted, label="predicted curve", color="#c0392b", linewidth=1.4, linestyle="--")
axes[1].set_title("Stage 5 — Loss: reconstructed tomogram at pixel (0,0)")
axes[1].set_xlabel("elevation / height (m)")
axes[1].set_ylabel("reflectivity (a.u.)")
axes[1].legend(loc="upper right")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_05_loss.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_05_loss.pdf")

In [ ]:
total_loss_value = float(loss_output_dictionary["total_loss"])
expected_curve_shape = (BATCH_SIZE, SPECTRAL_SAMPLE_POINTS, PATCH_HEIGHT, PATCH_WIDTH)

checks = [
    ("Output dict has required keys",           set(["total_loss", "components", "weighted"]).issubset(loss_output_dictionary.keys())),
    ("total_loss is a finite scalar",           bool(torch.isfinite(loss_output_dictionary["total_loss"])) and loss_output_dictionary["total_loss"].ndim == 0),
    ("total_loss is non-negative",              total_loss_value >= 0.0),
    ("Reconstructed curve shape (B,N,H,W)",     tuple(reconstructed_curves.shape) == expected_curve_shape),
    ("Identical pred and gt -> ~0 loss",        float(loss_output_identical["total_loss"]) < 1e-4),
    ("Active terms match enabled config flags", set(loss_output_dictionary["components"].keys()) >= {"mse_curve", "param_l1"}),
    ("Weighted term = raw x effective weight",  math.isclose(float(loss_output_dictionary["weighted"]["mse_curve"]), inspection_loss_config.eff("weight_mse_curve") * float(loss_output_dictionary["components"]["mse_curve"]), rel_tol=1e-4)),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 5: Loss Function

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Loss does not move despite training | a curve term disabled but assumed active; `weight_sum` divides it out | Print active terms vs `use_*` flags and `weighted` dict |
| Loss scale orders of magnitude off | `norm` normalization factors not applied (`eff` not used) | Compare `weighted[term]` to `α·norm·component` |
| param-L1 NaN with mismatched K | gt has fewer Gaussians than prediction; matcher slice misaligned | Inspect `effective_gaussians = min(K_pred, K_gt)` |
| Identical pred/gt gives non-zero loss | denormalize/normalize round-trip not identity, or clamp altering params | Run pred==gt and assert ~0; inspect `clamp_gaussian_params` |
| Curve term dominated by amplitude | `param_weights` not applied per (amp, mu, sigma) channel | Print per-param `param_l1/*` entries |
| Weights not changing on curriculum | `set_curriculum` not called or wrong config swapped | Compare `loss_cfg` before/after Stage 10 swap |

**Passing to Stage 6:** `loss_function` — `Loss`, the `criterion` consumed by `TrainStep`; `loss_output_dictionary` — the per-step dict fed to `MetricAggregator`.

---
## Stage 6: Single Training Step

**Callable:** `TrainStep.step(images, gt_params, batch_idx, n_batches, global_step)`
**Input:** one batch of SAR patches and gt parameters, the batch index, batches-per-epoch, and the global step
**Output:** `(loss, loss_dict)`; as a side effect it runs backward, (on an accumulation boundary) unscales, clips gradients, steps the optimizer, zeroes grads, and updates EMA.

The mini-batch loss is divided by the accumulation count $A$ before backward:

$$
\mathcal{L}_{\text{accum}} = \frac{\mathcal{L}_{\text{total}}}{A}
$$

and the optimizer steps when $(i+1) \bmod A = 0$ or $i+1 = n_{\text{batches}}$.

> **What you should see:** A finite scalar loss; after the step at least one model
> parameter has changed (weights updated). With `accumulation_steps = 1` the optimizer
> steps every batch. Repeating the step on the *same* batch should reduce the loss
> (the model is learning that batch). Gradients are populated then zeroed.

In [ ]:
train_step_model = copy.deepcopy(reconstruction_model).to(COMPUTE_DEVICE)
train_step_optimizer = torch.optim.AdamW(train_step_model.parameters(), lr=1e-2)
gradient_scaler = torch.amp.GradScaler("cuda", enabled=False)

train_step_loss_config = LossConfig(use_mse_curve=True, weight_mse_curve=1.0)
train_step_loss = Loss(spectral_axis_tensor, NotebookLogger(), notebook_tracker, GAUSSIAN_CONFIG, train_step_loss_config, norm_stats=identity_normalizer)

train_step_clipper = GradientClipper(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
train_step_ema     = EMA(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
train_step_ema.init(train_step_model)

single_training_step = TrainStep(
    model              = train_step_model,
    optimizer          = train_step_optimizer,
    scaler             = gradient_scaler,
    criterion          = train_step_loss,
    grad_clipper       = train_step_clipper,
    ema                = train_step_ema,
    device             = COMPUTE_DEVICE,
    logger             = NotebookLogger(),
    tracker            = notebook_tracker,
    accumulation_steps = TRAINER_CONFIG.training.gradient_accumulation_steps,
    use_amp            = False,
    ema_every          = 1,
)

step_input_batch   = synthetic_input_batch.detach().clone()
step_target_params = ground_truth_parameters.detach().clone()

parameters_before = copy.deepcopy([parameter.detach().clone() for parameter in train_step_model.parameters()])

repeated_step_losses = []
for step_index in range(25):
    loss_value, loss_dictionary = single_training_step.step(step_input_batch, step_target_params, batch_idx=0, n_batches=1, global_step=step_index)
    repeated_step_losses.append(float(loss_value) * single_training_step.accumulation_steps)

parameters_after = [parameter.detach().clone() for parameter in train_step_model.parameters()]
repeated_step_losses = np.asarray(repeated_step_losses, dtype=np.float64)

In [ ]:
maximum_parameter_shift = max(float((after - before).abs().max()) for before, after in zip(parameters_before, parameters_after))

print("Loss at first step        :", float(repeated_step_losses[0]))
print("Loss at last step         :", float(repeated_step_losses[-1]))
print("Loss decreased            :", bool(repeated_step_losses[-1] < repeated_step_losses[0]))
print("Max parameter shift       :", maximum_parameter_shift)
print("Accumulation steps        :", single_training_step.accumulation_steps)
print("Returned loss is finite   :", bool(np.isfinite(repeated_step_losses).all()))

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4.2))
axis.plot(np.arange(repeated_step_losses.shape[0]), repeated_step_losses, color="#1f4e79", linewidth=1.8, marker="o", markersize=3)
axis.set_title("Stage 6 — Train Step: loss over repeated steps on one batch")
axis.set_xlabel("optimizer step")
axis.set_ylabel("MSE-curve loss (unitless)")
axis.set_yscale("log")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_06_train_step.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_06_train_step.pdf")

In [ ]:
checks = [
    ("Returned loss is a finite scalar",        bool(np.isfinite(repeated_step_losses).all())),
    ("Parameters changed after stepping",       maximum_parameter_shift > 0.0),
    ("Loss decreased over repeated steps",      float(repeated_step_losses[-1]) < float(repeated_step_losses[0])),
    ("loss_dict carries components",            "components" in loss_dictionary and len(loss_dictionary["components"]) > 0),
    ("Gradients zeroed after opt step",         all(parameter.grad is None or float(parameter.grad.abs().sum()) == 0.0 for parameter in train_step_model.parameters())),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 6: Train Step

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Loss flat, weights never change | optimizer step gated on wrong accumulation boundary; never reached | Print `(batch_idx+1) % accumulation_steps`; confirm step fires |
| Effective LR too high under accumulation | loss not divided by `A`, so gradients are `A`× too large | Verify `loss / accumulation_steps` before backward |
| Grads not zeroed -> blow-up | `zero_grad` skipped on non-boundary batches (expected) but never on boundary | Check grads ≈ 0 right after an optimizer step |
| EMA updated every micro-batch | `ema.update` not gated by accumulation / `ema_every` | Confirm update only on `global_step % ema_every == 0` |
| Silent NaN under AMP | bf16 autocast with un-scaled overflow path | Enable `tracker.debug` to catch NaN/Inf warnings in forward |

**Passing to Stage 7:** `train_step_model` — `nn.Module` with populated gradients during a step; the gradient clipper operates between backward and optimizer step.

---
## Stage 7: Gradient Clipping

**Callable:** `GradientClipper.maybe_clip(model, global_step)` then `GradientClipper.record(norm, step)`
**Input:** a model whose parameters carry populated `.grad`
**Output:** the (possibly reduced) global gradient L2 norm. Gradients are rescaled in place when the norm exceeds the threshold.

The global norm and the rescale are

$$
\|\mathbf g\|_2 = \sqrt{\sum_p \|\mathbf g_p\|_2^2}, \qquad
\mathbf g \leftarrow \mathbf g \cdot \min\!\left(1, \frac{g_{\max}}{\|\mathbf g\|_2 + \epsilon}\right)
$$

> **What you should see:** For `clip_mode = "fixed"` with `max_grad_norm = 1.0`, any
> batch whose pre-clip norm exceeds 1.0 is scaled so its post-clip norm equals ~1.0;
> batches already under threshold pass through unchanged. The post-clip norm never
> exceeds the threshold. The clip ratio = `norm_after / norm_before` is ≤ 1.

In [ ]:
gradient_inspection_model = copy.deepcopy(reconstruction_model).to(COMPUTE_DEVICE)
gradient_clipper = GradientClipper(TRAINER_CONFIG, notebook_logger, notebook_tracker)

clip_threshold = TRAINER_CONFIG.gradient_clipper.max_grad_norm

pre_clip_norms  = []
post_clip_norms = []
for synthetic_loss_scale in np.linspace(0.2, 6.0, 30):
    gradient_inspection_model.zero_grad(set_to_none=True)
    model_prediction = gradient_inspection_model(synthetic_input_batch)
    synthetic_loss   = (model_prediction.pow(2).mean()) * float(synthetic_loss_scale)
    synthetic_loss.backward()

    norm_before = GradientClipper.global_norm(gradient_inspection_model)
    norm_after  = gradient_clipper.maybe_clip(gradient_inspection_model, global_step=0)

    pre_clip_norms.append(norm_before)
    post_clip_norms.append(norm_after)

pre_clip_norms  = np.asarray(pre_clip_norms, dtype=np.float64)
post_clip_norms = np.asarray(post_clip_norms, dtype=np.float64)

In [ ]:
print("Clip mode                 :", gradient_clipper.mode)
print("Fixed threshold g_max     :", clip_threshold)
print("Pre-clip norm  range      :", (float(pre_clip_norms.min()),  float(pre_clip_norms.max())))
print("Post-clip norm range      :", (float(post_clip_norms.min()), float(post_clip_norms.max())))
print("Count exceeding threshold :", int(np.sum(pre_clip_norms > clip_threshold)))
print("Max post-clip norm        :", float(post_clip_norms.max()))
print("Post-clip <= threshold    :", bool(np.all(post_clip_norms <= clip_threshold + 1e-4)))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].scatter(pre_clip_norms, post_clip_norms, color="#1f4e79", s=18)
axes[0].axhline(clip_threshold, color="#c0392b", linestyle="--", linewidth=1.2, label="g_max threshold")
axes[0].plot([0, pre_clip_norms.max()], [0, pre_clip_norms.max()], color="0.6", linewidth=1.0, label="identity (no clip)")
axes[0].set_title("Stage 7 — Gradient Clipper: pre vs post-clip norm")
axes[0].set_xlabel("pre-clip gradient L2 norm")
axes[0].set_ylabel("post-clip gradient L2 norm")
axes[0].legend(loc="upper left")

axes[1].hist(pre_clip_norms,  bins=15, alpha=0.65, label="before clip", color="#1f4e79", edgecolor="white")
axes[1].hist(post_clip_norms, bins=15, alpha=0.65, label="after clip",  color="#8c6d31", edgecolor="white")
axes[1].axvline(clip_threshold, color="#c0392b", linestyle="--", linewidth=1.2)
axes[1].set_title("Stage 7 — Gradient Clipper: norm distribution")
axes[1].set_xlabel("gradient L2 norm")
axes[1].set_ylabel("count")
axes[1].legend(loc="upper right")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_07_gradient_clipper.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_07_gradient_clipper.pdf")

In [ ]:
unclipped_unchanged = bool(np.all(np.isclose(pre_clip_norms[pre_clip_norms <= clip_threshold], post_clip_norms[pre_clip_norms <= clip_threshold], atol=1e-4)))

checks = [
    ("Post-clip norm never exceeds threshold",  bool(np.all(post_clip_norms <= clip_threshold + 1e-4))),
    ("At least one batch was clipped",          int(np.sum(pre_clip_norms > clip_threshold)) > 0),
    ("Sub-threshold norms pass through",        unclipped_unchanged),
    ("All norms finite and non-negative",       bool(np.all(np.isfinite(pre_clip_norms)) and np.all(pre_clip_norms >= 0.0))),
    ("Clip mode taken from config",             gradient_clipper.mode == TRAINER_CONFIG.gradient_clipper.clip_mode),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 7: Gradient Clipping

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Clipping has no effect | clip done *after* optimizer step, or before `scaler.unscale_` under AMP | Confirm order: backward → unscale → clip → step |
| Norm reported but never reduced | `clip_mode = "disabled"`, or adaptive window not yet filled (returns `None`) | Print `mode` and `len(history)` vs `adaptive_window` |
| Threshold in wrong units | confusing per-parameter norm with global norm | `global_norm` stacks per-param norms then takes L2 of the stack |
| Adaptive threshold too loose early | first `window` steps unclipped while history fills | Expect no clip until `len(history) >= window` |
| Exploding norm warning spam | LR too high or bad init feeding huge grads | Cross-check with Stage 2 init mode and Stage 4 LR |

**Passing to Stage 8:** the clipped gradients feed the optimizer step; immediately after, `EMA.update` blends the updated weights into the shadow.

---
## Stage 8: Exponential Moving Average (EMA) Update

**Callable:** `EMA.update(model, step)`, `EMA.apply_to(model)`, `EMA.restore(model)`
**Input:** the live model parameters after each optimizer step
**Output:** shadow parameters `θ̃` blended from the live weights; `apply_to` swaps them in for evaluation and `restore` swaps the live weights back.

The shadow update is

$$
\tilde\theta_t = \gamma\,\tilde\theta_{t-1} + (1-\gamma)\,\theta_t, \qquad \gamma \in [0,1)
$$

> **What you should see:** Starting from a copy of the live weights, after many update
> steps the shadow lags the live weights — the divergence
> `Σ ‖θ̃ - θ‖₂` grows then stabilizes. With high `γ` (= 0.999) the shadow moves slowly.
> `apply_to` then `restore` must leave the live weights bit-identical to before the swap.

In [ ]:
ema_inspection_model = copy.deepcopy(reconstruction_model).to(COMPUTE_DEVICE)
ema_controller = EMA(TRAINER_CONFIG, notebook_logger, notebook_tracker)
ema_controller.init(ema_inspection_model)

ema_decay = ema_controller.decay
divergence_trace = []
with torch.no_grad():
    for update_step in range(150):
        for parameter in ema_inspection_model.parameters():
            if parameter.requires_grad:
                parameter.add_(0.01 * torch.randn_like(parameter))
        ema_controller.update(ema_inspection_model, step=update_step)
        divergence = sum(float(torch.norm(ema_controller.shadow[name] - parameter)) for name, parameter in ema_inspection_model.named_parameters() if parameter.requires_grad)
        divergence_trace.append(divergence)

divergence_trace = np.asarray(divergence_trace, dtype=np.float64)

weights_before_swap = [parameter.detach().clone() for parameter in ema_inspection_model.parameters()]
ema_controller.apply_to(ema_inspection_model)
weights_during_swap = [parameter.detach().clone() for parameter in ema_inspection_model.parameters()]
ema_controller.restore(ema_inspection_model)
weights_after_restore = [parameter.detach().clone() for parameter in ema_inspection_model.parameters()]

In [ ]:
maximum_restore_error = max(float((before - after).abs().max()) for before, after in zip(weights_before_swap, weights_after_restore))
maximum_swap_shift    = max(float((before - during).abs().max()) for before, during in zip(weights_before_swap, weights_during_swap))

print("EMA enabled               :", ema_controller.enabled)
print("EMA decay gamma           :", ema_decay)
print("Divergence at step 0      :", float(divergence_trace[0]))
print("Divergence at final step  :", float(divergence_trace[-1]))
print("apply_to shifts weights   :", maximum_swap_shift > 0.0)
print("restore recovers weights  :", maximum_restore_error)

In [ ]:
def divergence_trace_for_decay(decay_value: float) -> np.ndarray:
    decay_config = copy.deepcopy(TRAINER_CONFIG)
    decay_config.ema.ema_decay = decay_value
    decay_model = copy.deepcopy(reconstruction_model).to(COMPUTE_DEVICE)
    decay_ema   = EMA(decay_config, NotebookLogger(), NotebookTracker())
    decay_ema.init(decay_model)
    torch.manual_seed(RANDOM_SEED)
    trace = []
    with torch.no_grad():
        for update_step in range(150):
            for parameter in decay_model.parameters():
                if parameter.requires_grad:
                    parameter.add_(0.01 * torch.randn_like(parameter))
            decay_ema.update(decay_model, step=update_step)
            trace.append(sum(float(torch.norm(decay_ema.shadow[name] - parameter)) for name, parameter in decay_model.named_parameters() if parameter.requires_grad))
    return np.asarray(trace, dtype=np.float64)

figure, axis = plt.subplots(figsize=(8, 4.2))
for decay_value, color in [(0.9, "#2e7d32"), (0.99, "#8c6d31"), (0.999, "#1f4e79")]:
    axis.plot(np.arange(150), divergence_trace_for_decay(decay_value), label=f"gamma = {decay_value}", linewidth=1.6, color=color)
axis.set_title("Stage 8 — EMA: shadow-vs-live divergence by decay")
axis.set_xlabel("EMA update step")
axis.set_ylabel(r"divergence  $\sum \|\tilde\theta - \theta\|_2$")
axis.legend(loc="upper left", title="ema_decay")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_08_ema.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_08_ema.pdf")

In [ ]:
checks = [
    ("EMA enabled matches config",              ema_controller.enabled == TRAINER_CONFIG.ema.use_ema),
    ("Shadow diverges from live weights",       float(divergence_trace[-1]) > 0.0),
    ("apply_to swaps in shadow weights",        maximum_swap_shift > 0.0),
    ("restore recovers live weights exactly",   maximum_restore_error < 1e-6),
    ("Decay within [0, 1)",                     0.0 <= ema_decay < 1.0),
    ("Shadow has one entry per trainable param",len(ema_controller.shadow) == sum(1 for parameter in ema_inspection_model.parameters() if parameter.requires_grad)),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 8: EMA

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Eval weights never restored | `restore` not called in a `finally`, leaving shadow in the live model | Diff weights before `apply_to` and after `restore` |
| EMA masks real divergence/instability | `ema_decay` too high (0.9999) so shadow ignores recent updates | Sweep decay; watch divergence growth rate |
| Shadow not updated | `ema_every` too large, or update gated off the accumulation boundary | Confirm `global_step % ema_every == 0` fires |
| Shadow missing params on resume | `ema_shadow` not restored from checkpoint, or buffers excluded | Check `state_dict()['shadow']` keys vs trainable params |
| Eval better than train but model bad | reporting EMA metrics while saving raw weights (or vice versa) | Verify checkpoint stores both raw and `ema_shadow` |

**Passing to Stage 9:** during evaluation the EMA-applied model produces `loss_dict`s; those are summed by the metric aggregator.

---
## Stage 9: Metric Aggregation

**Callable:** `MetricAggregator.add(loss_dict)` per batch, then `reduce_components()` / `reduce_weighted()` / `reduce_extra()`
**Input:** the per-batch `loss_dict` (and optional extra permutation metrics)
**Output:** epoch-mean dictionaries — each accumulated term divided by the batch count.

For each term $i$ over $M$ batches,

$$
\bar\ell_i = \frac{1}{M}\sum_{m=1}^{M} \ell_i^{(m)}
$$

> **What you should see:** After adding `M` batches, `reduce_components()[term]` equals
> the arithmetic mean of that term across the batches, and `count == M`. Keys are
> exactly the union of component keys seen. NaNs in extras are coerced to 0 before
> summation (guarding the running totals).

In [ ]:
metric_aggregator = MetricAggregator()

number_of_batches = 12
manual_component_totals = {}
for batch_number in range(number_of_batches):
    mse_value     = float(0.5 + 0.1 * batch_number)
    param_value   = float(0.2 + 0.05 * batch_number)
    synthetic_loss_dict = {
        "total_loss" : torch.tensor(mse_value + param_value),
        "components" : {"mse_curve": torch.tensor(mse_value), "param_l1": torch.tensor(param_value)},
        "weighted"   : {"mse_curve": torch.tensor(mse_value * 1.0), "param_l1": torch.tensor(param_value * 0.1)},
    }
    metric_aggregator.add(synthetic_loss_dict)
    for term, value in synthetic_loss_dict["components"].items():
        manual_component_totals[term] = manual_component_totals.get(term, 0.0) + float(value)

reduced_components = metric_aggregator.reduce_components()
reduced_weighted   = metric_aggregator.reduce_weighted()
manual_component_means = {term: total / number_of_batches for term, total in manual_component_totals.items()}

In [ ]:
print("Batches aggregated        :", metric_aggregator.count)
print()
print("Reduced components (mean):")
for term, value in reduced_components.items():
    print(f"    {term:>14} = {value:.6f}   (manual mean = {manual_component_means[term]:.6f})")
print()
print("Reduced weighted (mean):")
for term, value in reduced_weighted.items():
    print(f"    {term:>14} = {value:.6f}")

In [ ]:
term_labels   = list(reduced_components.keys())
reduced_values = [reduced_components[term] for term in term_labels]
manual_values  = [manual_component_means[term] for term in term_labels]

figure, axis = plt.subplots(figsize=(7.5, 4.2))
bar_positions = np.arange(len(term_labels))
bar_width = 0.38
axis.bar(bar_positions - bar_width / 2, reduced_values, bar_width, label="aggregator mean", color="#1f4e79", edgecolor="white")
axis.bar(bar_positions + bar_width / 2, manual_values,  bar_width, label="manual mean",     color="#8c6d31", edgecolor="white")
axis.set_xticks(bar_positions)
axis.set_xticklabels(term_labels)
axis.set_title("Stage 9 — Metric Aggregator: epoch-mean per component")
axis.set_xlabel("loss component")
axis.set_ylabel("epoch-mean value (unitless)")
axis.legend(loc="upper left")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_09_metric_aggregator.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_09_metric_aggregator.pdf")

In [ ]:
nan_coerced_aggregator = MetricAggregator()
nan_coerced_aggregator.add_extra({"perm_margin": float("nan"), "perm_rmse": 1.5})

checks = [
    ("Batch count equals batches added",        metric_aggregator.count == number_of_batches),
    ("Component mean matches manual mean",       all(math.isclose(reduced_components[term], manual_component_means[term], rel_tol=1e-9) for term in term_labels)),
    ("Reduced keys match input component keys",  set(reduced_components.keys()) == set(manual_component_means.keys())),
    ("Weighted mean = component mean x weight",  math.isclose(reduced_weighted["param_l1"], reduced_components["param_l1"] * 0.1, rel_tol=1e-9)),
    ("NaN extras coerced to zero in sum",        math.isclose(nan_coerced_aggregator.extra_sum["perm_margin"], 0.0)),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 9: Metric Aggregation

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Reported loss is a sum, not a mean | `reduce_*` not called, or divided by wrong count | Confirm division by `count`, and `count` increments once per `add` |
| Epoch metric biased by last batch size | uneven final batch weighted equally (mean of means) | Note: aggregator means per-batch values, not per-sample |
| NaN poisons the whole metric | extra metric NaN added without the `v == v` guard | Inspect `add_extra` coercion; assert finite reduced extras |
| Component keys drift between epochs | curriculum swap changes active terms mid-run | Compare key sets before/after Stage 10 |
| Weighted ≠ component × weight | weighting applied twice or with stale `eff` | Cross-check `weighted[term]` vs `component[term]·w` |

**Passing to Stage 10:** the aggregated metrics summarize an epoch; at epoch boundaries the curriculum controller may swap the loss configuration that defines those components.

---
## Stage 10: Curriculum Loss Swap

**Callable:** `CurriculumController.maybe_swap(epoch)`
**Input:** the current epoch index
**Output:** at `swap_epoch` (when enabled), the criterion's loss config is replaced by `curriculum.complete`, and optionally early-stopping, LR scheduler, warmup, and optimizer state are reset.

The swap is gated by

$$
\text{swap}(t) = \text{enabled} \;\wedge\; (t = t_{\text{swap}})
$$

> **What you should see:** Before `swap_epoch`, `maybe_swap` returns `False` and the
> loss config is unchanged. At exactly `swap_epoch` it returns `True`, the criterion's
> `loss_cfg` becomes the `complete` config, and the `ParamMatcher.strategy` updates to
> the complete config's `param_match`. Reset flags fire only if configured.

In [ ]:
warmup_loss_config   = LossConfig(use_mse_curve=True, weight_mse_curve=1.0, param_match="sort_gt_by_mu")
complete_loss_config = LossConfig(use_mse_curve=True, weight_mse_curve=1.0, use_param_l1=True, weight_param_l1=0.1, use_ssim_curve=True, weight_ssim_curve=0.5, param_match="hungarian_mu")

curriculum_configuration = LossCurriculumConfig(
    enabled              = True,
    swap_epoch           = 3,
    warmup               = warmup_loss_config,
    complete             = complete_loss_config,
    reset_early_stopping = True,
    reset_lr             = True,
    reset_warmup         = True,
)

curriculum_criterion = Loss(spectral_axis_tensor, NotebookLogger(), notebook_tracker, GAUSSIAN_CONFIG, warmup_loss_config, norm_stats=identity_normalizer)
curriculum_warmup    = Warmup(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
curriculum_scheduler = Scheduler(base_learning_rates, curriculum_warmup, TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
curriculum_early_stop = EarlyStopping(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
curriculum_optimizer  = torch.optim.AdamW(copy.deepcopy(reconstruction_model).parameters(), lr=1e-3)

applied_learning_rates = {}
def record_learning_rate_update(learning_rates):
    applied_learning_rates["last"] = list(learning_rates)

curriculum_controller = CurriculumController(
    curriculum       = curriculum_configuration,
    criterion        = curriculum_criterion,
    early_stopping   = curriculum_early_stop,
    lr_scheduler     = curriculum_scheduler,
    warmup           = curriculum_warmup,
    optimizer        = curriculum_optimizer,
    update_optimizer = record_learning_rate_update,
    logger           = NotebookLogger(),
)

active_terms_before_swap = sorted(name for name, used in [
    ("mse_curve",  curriculum_criterion.loss_cfg.use_mse_curve),
    ("param_l1",   curriculum_criterion.loss_cfg.use_param_l1),
    ("ssim_curve", curriculum_criterion.loss_cfg.use_ssim_curve),
] if used)
strategy_before_swap = curriculum_criterion.matcher.strategy

swap_decision_trace = [curriculum_controller.maybe_swap(epoch) for epoch in range(6)]

active_terms_after_swap = sorted(name for name, used in [
    ("mse_curve",  curriculum_criterion.loss_cfg.use_mse_curve),
    ("param_l1",   curriculum_criterion.loss_cfg.use_param_l1),
    ("ssim_curve", curriculum_criterion.loss_cfg.use_ssim_curve),
] if used)
strategy_after_swap = curriculum_criterion.matcher.strategy

In [ ]:
print("Swap epoch                :", curriculum_configuration.swap_epoch)
print("maybe_swap per epoch      :", swap_decision_trace)
print("Active terms before swap  :", active_terms_before_swap)
print("Active terms after swap   :", active_terms_after_swap)
print("Matcher strategy before   :", strategy_before_swap)
print("Matcher strategy after    :", strategy_after_swap)
print("Warmup reset (step 0)     :", curriculum_warmup.current_step == 0)
print("Early stopping reset      :", curriculum_early_stop.best_loss is None)

In [ ]:
swap_indicator = np.zeros(len(swap_decision_trace))
for epoch_index, decision in enumerate(swap_decision_trace):
    swap_indicator[epoch_index] = 1.0 if decision else 0.0

figure, axis = plt.subplots(figsize=(7.5, 3.8))
axis.step(np.arange(len(swap_decision_trace)), swap_indicator, where="mid", color="#1f4e79", linewidth=1.8)
axis.axvline(curriculum_configuration.swap_epoch, color="#c0392b", linestyle="--", linewidth=1.2, label="swap_epoch")
axis.set_ylim(-0.2, 1.2)
axis.set_yticks([0, 1])
axis.set_yticklabels(["no swap", "swap fired"])
axis.set_title("Stage 10 — Curriculum: loss swap trigger by epoch")
axis.set_xlabel("epoch t")
axis.set_ylabel("swap decision")
axis.legend(loc="center right")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_10_curriculum.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_10_curriculum.pdf")

In [ ]:
checks = [
    ("No swap before swap_epoch",               not any(swap_decision_trace[: curriculum_configuration.swap_epoch])),
    ("Swap fires exactly at swap_epoch",        swap_decision_trace[curriculum_configuration.swap_epoch] is True),
    ("Swap fires only once",                    sum(1 for decision in swap_decision_trace if decision) == 1),
    ("Loss config replaced by complete cfg",    active_terms_after_swap == sorted(["mse_curve", "param_l1", "ssim_curve"])),
    ("Matcher strategy updated to complete",    strategy_after_swap == complete_loss_config.param_match),
    ("Warmup reset on swap",                    curriculum_warmup.current_step == 0),
    ("Early stopping reset on swap",            curriculum_early_stop.best_loss is None),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 10: Curriculum Swap

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| New loss terms never activate | `curriculum.enabled` False, or `swap_epoch` past total epochs | Print `enabled`, `swap_epoch` vs `epochs` |
| Swap fires every epoch | comparison `epoch >= swap_epoch` instead of `==` | Confirm trace has a single `True` |
| LR unchanged after a `reset_lr` swap | `_epoch_offset` not applied, or `update_optimizer` not invoked | Check `Scheduler._epoch_offset` and recorded LRs |
| Optimizer keeps stale Adam moments | `reset_optimizer` False when a sharp loss change is expected | Inspect optimizer `state` cleared per param |
| Param matcher mismatched after swap | `set_curriculum` updates `loss_cfg` but not `matcher.strategy` | Verify both `loss_cfg` and `matcher.strategy` changed |

**Passing to Stage 11:** after each evaluated epoch the (possibly swapped) criterion yields a validation loss that drives early stopping.

---
## Stage 11: Early Stopping Check

**Callable:** `EarlyStopping.__call__(val_loss, model, epoch)`
**Input:** the per-eval validation loss, the model, and the epoch index
**Output:** a boolean stop flag; internally tracks the best loss, a no-improvement counter, and (when `restore_best`) a snapshot of the best weights.

Stopping triggers when no improvement of at least $\delta_{\min}$ occurs for $P$ consecutive evaluations:

$$
\text{stop}(t) = \big[\, \ell_{\text{val}}(t') \ge \ell^{*}_{\text{val}} - \delta_{\min}\ \ \forall\, t' \in \{t-P+1,\dots,t\} \,\big]
$$

> **What you should see:** On a strictly improving loss sequence the counter stays at 0
> and `stop` is always `False`. On a flat/worsening sequence the counter increments each
> eval and `stop` flips to `True` exactly when it reaches `patience` (= 15). The best
> loss equals the minimum seen, and the best epoch is its index.

In [ ]:
early_stopping_patience = TRAINER_CONFIG.early_stopping.patience
early_stopping_min_delta = TRAINER_CONFIG.early_stopping.min_delta

improving_early_stop = EarlyStopping(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
improving_losses = [1.0 - 0.02 * epoch for epoch in range(early_stopping_patience + 5)]
improving_counter_trace = []
improving_stop_trace = []
for epoch_index, validation_loss in enumerate(improving_losses):
    stop_flag = improving_early_stop(validation_loss, reconstruction_model, epoch_index)
    improving_counter_trace.append(improving_early_stop.counter)
    improving_stop_trace.append(stop_flag)

plateau_early_stop = EarlyStopping(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
plateau_losses = [0.5] + [0.5 + 0.001 * epoch for epoch in range(early_stopping_patience + 5)]
plateau_counter_trace = []
plateau_stop_trace = []
for epoch_index, validation_loss in enumerate(plateau_losses):
    stop_flag = plateau_early_stop(validation_loss, reconstruction_model, epoch_index)
    plateau_counter_trace.append(plateau_early_stop.counter)
    plateau_stop_trace.append(stop_flag)

first_stop_epoch = next((index for index, flag in enumerate(plateau_stop_trace) if flag), None)

In [ ]:
print("Patience P                :", early_stopping_patience)
print("Min delta                 :", early_stopping_min_delta)
print()
print("Improving sequence -> any stop :", any(improving_stop_trace))
print("Improving best loss            :", improving_early_stop.best_loss)
print()
print("Plateau sequence first stop epoch:", first_stop_epoch)
print("Plateau counter at stop          :", plateau_counter_trace[first_stop_epoch] if first_stop_epoch is not None else None)
print("Plateau best loss                :", plateau_early_stop.best_loss)
print("Plateau best epoch               :", plateau_early_stop.best_epoch)

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4.2))
axis.plot(np.arange(len(improving_counter_trace)), improving_counter_trace, label="improving loss", color="#2e7d32", linewidth=1.8, marker="o", markersize=3)
axis.plot(np.arange(len(plateau_counter_trace)),  plateau_counter_trace,  label="plateaued loss", color="#c0392b", linewidth=1.8, marker="s", markersize=3)
axis.axhline(early_stopping_patience, color="0.4", linestyle="--", linewidth=1.2, label="patience")
axis.set_title("Stage 11 — Early Stopping: no-improvement counter")
axis.set_xlabel("evaluation index")
axis.set_ylabel("consecutive no-improvement count")
axis.legend(loc="upper left")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_11_early_stopping.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_11_early_stopping.pdf")

In [ ]:
checks = [
    ("No stop while loss improves",             not any(improving_stop_trace)),
    ("Counter stays 0 while improving",         max(improving_counter_trace) == 0),
    ("Stop fires on plateau",                   any(plateau_stop_trace)),
    ("Stop fires at patience count",            (first_stop_epoch is not None) and (plateau_counter_trace[first_stop_epoch] == early_stopping_patience)),
    ("Best loss equals minimum seen",           math.isclose(improving_early_stop.best_loss, min(improving_losses), rel_tol=1e-9)),
    ("triggered flag set after stop",           plateau_early_stop.triggered),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 11: Early Stopping

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Stops far too early | patience counted per eval but evals are infrequent (`validation_frequency` large) | Compare effective patience in epochs = `patience × validation_frequency` |
| Never stops on a plateau | `min_delta` too large so tiny gains reset the counter | Print counter trace; check `val_loss < best - min_delta` |
| "Improvement" on noisy loss resets counter | `min_delta` too small (0) for noisy validation | Inspect best-loss updates against noise scale |
| Best weights not restored | `restore_best` False, or `best_params` never saved | Confirm `_save_state` ran and `restore` loads it |
| Counter survives curriculum swap | `reset_early_stopping` not set when a regime change is expected | Check Stage 10 reset flags |

**Passing to Stage 12:** the same validation loss that drives early stopping also drives checkpointing — a new best loss writes a checkpoint.

---
## Stage 12: Checkpoint Save

**Callable:** `Checkpoint.step(val_loss, epoch, trainer)` → `Checkpoint.save / load`
**Input:** the validation loss, the epoch, and a trainer exposing `capture_state` / `restore_state`
**Output:** on a new-best loss, a `best_model.pt` containing model params, optimizer, scheduler, warmup, EMA, and early-stopping state; `load` round-trips them.

A checkpoint is written iff

$$
\ell_{\text{val}}(t) < \ell^{*}_{\text{val}}
$$

> **What you should see:** A checkpoint file appears only when the loss strictly
> improves; a worse loss is a no-op. The saved dict carries optimizer, scheduler,
> warmup, EMA, and early-stopping state (so resume is exact). Loading restores
> `global_step`, the LR scheduler's `current_lrs`, the warmup `current_step`, and the
> EMA shadow — not just the model weights.

In [ ]:
class MinimalTrainerState:
    def __init__(self, model, optimizer, scheduler, warmup, ema, early_stopping, x_axis_tensor):
        self.model          = model
        self.optimizer      = optimizer
        self.lr_scheduler   = scheduler
        self.warmup         = warmup
        self.ema            = ema
        self.early_stopping = early_stopping
        self.x_axis         = x_axis_tensor
        self.global_step    = 1234
        self.train_losses   = [0.9, 0.7, 0.5]
        self.val_losses     = [0.95, 0.72, 0.55]
        self.config         = TRAINER_CONFIG

    def capture_state(self, epoch):
        return {
            "epoch"                : epoch,
            "global_step"          : self.global_step,
            "train_losses"         : self.train_losses,
            "val_losses"           : self.val_losses,
            "params"               : self.model.state_dict(),
            "opt_state"            : self.optimizer.state_dict(),
            "batch_stats"          : None,
            "ema_shadow"           : self.ema.state_dict() if self.ema is not None else None,
            "config"               : str(self.config),
            "x_axis"               : self.x_axis.cpu().numpy(),
            "scheduler_state"      : self.lr_scheduler.state_dict(),
            "warmup_state"         : self.warmup.state_dict(),
            "early_stopping_state" : self.early_stopping.state_dict(),
        }

    def restore_state(self, checkpoint):
        self.model.load_state_dict(checkpoint["params"])
        self.optimizer.load_state_dict(checkpoint["opt_state"])
        if checkpoint.get("ema_shadow") is not None and self.ema.enabled:
            self.ema.load_state_dict(checkpoint["ema_shadow"])
        self.global_step  = checkpoint["global_step"]
        self.train_losses = checkpoint["train_losses"]
        self.val_losses   = checkpoint["val_losses"]
        self.lr_scheduler.load_state_dict(checkpoint["scheduler_state"])
        self.warmup.load_state_dict(checkpoint["warmup_state"])
        self.early_stopping.load_state_dict(checkpoint["early_stopping_state"])
        return checkpoint["epoch"]


checkpoint_directory = FIGURE_DIRECTORY.parent.parent / "_checkpoint_inspection"
checkpoint_directory.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_directory / "best_model.pt"

checkpoint_model     = copy.deepcopy(reconstruction_model).to(COMPUTE_DEVICE)
checkpoint_optimizer = torch.optim.AdamW(checkpoint_model.parameters(), lr=1e-3)
checkpoint_warmup    = Warmup(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
for _ in range(50):
    checkpoint_warmup.step()
checkpoint_scheduler = Scheduler(base_learning_rates, checkpoint_warmup, TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
checkpoint_scheduler.step(5)
checkpoint_ema       = EMA(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
checkpoint_ema.init(checkpoint_model)
checkpoint_early_stop = EarlyStopping(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
checkpoint_early_stop(0.42, checkpoint_model, 5)

minimal_trainer = MinimalTrainerState(checkpoint_model, checkpoint_optimizer, checkpoint_scheduler, checkpoint_warmup, checkpoint_ema, checkpoint_early_stop, spectral_axis_tensor)

checkpoint_handler = Checkpoint(NotebookLogger(), notebook_tracker, str(checkpoint_path))

improvement_saved   = checkpoint_handler.step(val_loss=0.42, epoch=5, trainer=minimal_trainer)
no_improvement_saved = checkpoint_handler.step(val_loss=0.50, epoch=6, trainer=minimal_trainer)
checkpoint_exists    = checkpoint_path.exists()

global_step_before   = minimal_trainer.global_step
scheduler_lrs_before = list(checkpoint_scheduler.current_lrs)
warmup_step_before   = checkpoint_warmup.current_step

minimal_trainer.global_step = 0
checkpoint_handler.load(minimal_trainer, str(checkpoint_path))
global_step_after    = minimal_trainer.global_step
scheduler_lrs_after  = list(minimal_trainer.lr_scheduler.current_lrs)
warmup_step_after    = minimal_trainer.warmup.current_step

In [ ]:
saved_checkpoint = torch.load(str(checkpoint_path), weights_only=False)

print("Improvement triggered save :", improvement_saved)
print("Non-improvement saved      :", no_improvement_saved)
print("Checkpoint file exists     :", checkpoint_exists)
print()
print("Checkpoint payload keys    :")
for key in sorted(saved_checkpoint.keys()):
    print(f"    {key}")
print()
print("global_step  before/after restore :", global_step_before, "->", global_step_after)
print("scheduler LR before/after restore :", scheduler_lrs_before, "->", scheduler_lrs_after)
print("warmup step  before/after restore :", warmup_step_before, "->", warmup_step_after)

In [ ]:
required_state_keys = ["params", "opt_state", "scheduler_state", "warmup_state", "ema_shadow", "early_stopping_state", "global_step", "x_axis"]
present_state_keys  = [key for key in required_state_keys if key in saved_checkpoint]

figure, axis = plt.subplots(figsize=(8, 4.0))
key_present_flags = [1 if key in saved_checkpoint else 0 for key in required_state_keys]
axis.barh(range(len(required_state_keys)), key_present_flags, color="#1f4e79", edgecolor="white")
axis.set_yticks(range(len(required_state_keys)))
axis.set_yticklabels(required_state_keys)
axis.set_xlim(0, 1.2)
axis.set_xticks([0, 1])
axis.set_xticklabels(["absent", "present"])
axis.set_title("Stage 12 — Checkpoint: persisted state components")
axis.set_xlabel("presence in checkpoint")
axis.set_ylabel("state component")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_12_checkpoint.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_12_checkpoint.pdf")

In [ ]:
checks = [
    ("Save fires on improvement",               improvement_saved is True),
    ("Save skipped without improvement",        no_improvement_saved is False),
    ("Checkpoint file written",                 checkpoint_exists),
    ("All resume-critical state persisted",     present_state_keys == required_state_keys),
    ("global_step restored",                    global_step_after == global_step_before),
    ("Scheduler LRs restored",                  scheduler_lrs_after == scheduler_lrs_before),
    ("Warmup step restored",                    warmup_step_after == warmup_step_before),
    ("EMA shadow present in checkpoint",        saved_checkpoint.get("ema_shadow") is not None),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 12: Checkpointing

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Resume restarts LR / warmup schedule | only `model_state_dict` saved; scheduler/warmup state dropped | List checkpoint keys; assert scheduler + warmup present |
| Optimizer momentum lost on resume | `opt_state` not captured/restored | Compare optimizer `state_dict()` before save and after load |
| EMA evaluation degrades after resume | `ema_shadow` not restored, shadow re-initialized to live weights | Confirm `ema_shadow` non-None and reloaded |
| Best checkpoint overwritten by worse | comparison uses `<=` not `<`, or best loss tracked elsewhere | Verify save only on strict improvement |
| `x_axis` mismatch at inference | spectral grid not stored, inference assumes a default | Check `x_axis` array persisted in checkpoint |

**Passing to Stage 13:** with the per-epoch loop fully covered, the overfit probe exercises the loop end-to-end on a single repeated batch as a capacity sanity check.

---
## Stage 13: Overfit Single-Batch Sanity Probe

**Callable:** `OverfitManager.setup_loaders(train, val, test)` and `OverfitManager.check_stop(train_loss)`; combined with a real `TrainStep` loop
**Input:** the data loaders and per-epoch train loss
**Output:** when enabled, a single batch is sampled and repeated for the whole epoch; training should drive the loss toward ~0 (the model can memorize one batch), confirming end-to-end wiring and model capacity.

The probe asserts the optimisation can reach

$$
\mathcal{L}_{\text{train}}(\text{single batch}) \to 0
$$

> **What you should see:** `setup_loaders` returns a `[single_batch] * epoch_steps` list
> for train and a single-element list for val/test. Repeatedly stepping on that one
> batch drives the loss down by orders of magnitude, approaching the `stop_threshold`.
> If the loss plateaus far above 0, capacity or wiring is wrong, not the data. The probe
> requires a **deterministic** batch — non-deterministic augmentation would defeat it.

In [ ]:
overfit_config_enabled = copy.deepcopy(TRAINER_CONFIG)
overfit_config_enabled.overfit = OverfitConfig(enabled=True, max_steps=200, stop_threshold=1e-4, batch_size=BATCH_SIZE)

overfit_manager = OverfitManager(overfit_config_enabled, notebook_logger)

class StaticBatchLoader:
    def __init__(self, images, targets, length):
        self.images  = images
        self.targets = targets
        self.length  = length
    def __iter__(self):
        for _ in range(self.length):
            yield (self.images, self.targets)
    def __len__(self):
        return self.length

static_loader = StaticBatchLoader(synthetic_input_batch.cpu(), ground_truth_parameters.cpu(), 8)
overfit_train_loader, overfit_val_loader, overfit_test_loader = overfit_manager.setup_loaders(static_loader, static_loader, static_loader)

overfit_model = copy.deepcopy(reconstruction_model).to(COMPUTE_DEVICE)
overfit_optimizer = torch.optim.AdamW(overfit_model.parameters(), lr=1e-2)
overfit_loss = Loss(spectral_axis_tensor, NotebookLogger(), notebook_tracker, GAUSSIAN_CONFIG, LossConfig(use_mse_curve=True, weight_mse_curve=1.0), norm_stats=identity_normalizer)
overfit_clipper = GradientClipper(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
overfit_ema = EMA(TRAINER_CONFIG, NotebookLogger(), notebook_tracker)
overfit_ema.init(overfit_model)
overfit_scaler = torch.amp.GradScaler("cuda", enabled=False)

overfit_train_step = TrainStep(
    model=overfit_model, optimizer=overfit_optimizer, scaler=overfit_scaler, criterion=overfit_loss,
    grad_clipper=overfit_clipper, ema=overfit_ema, device=COMPUTE_DEVICE, logger=NotebookLogger(),
    tracker=notebook_tracker, accumulation_steps=1, use_amp=False, ema_every=1,
)

fixed_images, fixed_targets = overfit_train_loader[0]
fixed_images  = fixed_images.to(COMPUTE_DEVICE)
fixed_targets = fixed_targets.to(COMPUTE_DEVICE)

overfit_loss_trace = []
for step_index in range(150):
    loss_value, _ = overfit_train_step.step(fixed_images, fixed_targets, batch_idx=0, n_batches=1, global_step=step_index)
    overfit_loss_trace.append(float(loss_value))

overfit_loss_trace = np.asarray(overfit_loss_trace, dtype=np.float64)
overfit_should_stop = overfit_manager.check_stop(float(overfit_loss_trace[-1]))

In [ ]:
print("Overfit enabled            :", overfit_manager.enabled)
print("Repeated train-loader len  :", len(overfit_train_loader))
print("Val / test loader len      :", len(overfit_val_loader), "/", len(overfit_test_loader))
print("Stop threshold             :", overfit_config_enabled.overfit.stop_threshold)
print("Loss at step 0             :", float(overfit_loss_trace[0]))
print("Loss at final step         :", float(overfit_loss_trace[-1]))
print("Loss reduction factor      :", float(overfit_loss_trace[0] / max(overfit_loss_trace[-1], 1e-12)))
print("check_stop on final loss   :", overfit_should_stop)

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4.4))
axis.plot(np.arange(overfit_loss_trace.shape[0]), overfit_loss_trace, color="#1f4e79", linewidth=1.8)
axis.axhline(overfit_config_enabled.overfit.stop_threshold, color="#c0392b", linestyle="--", linewidth=1.2, label="stop_threshold")
axis.set_yscale("log")
axis.set_title("Stage 13 — Overfit Probe: single-batch loss landscape")
axis.set_xlabel("optimizer step on the repeated batch")
axis.set_ylabel("MSE-curve loss (log scale)")
axis.legend(loc="upper right")

figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / "stage_13_overfit_probe.pdf")
plt.show()
print("Saved figure -> notebooks/figures/inspect_training_pipeline/stage_13_overfit_probe.pdf")

In [ ]:
overfit_disabled_manager = OverfitManager(TRAINER_CONFIG, NotebookLogger())
disabled_train, disabled_val, disabled_test = overfit_disabled_manager.setup_loaders(static_loader, static_loader, static_loader)

checks = [
    ("Disabled overfit passes loaders through", disabled_train is static_loader),
    ("Enabled overfit repeats single batch",    len(overfit_train_loader) == min(len(static_loader), overfit_config_enabled.overfit.max_steps)),
    ("Val and test reduced to one batch",       len(overfit_val_loader) == 1 and len(overfit_test_loader) == 1),
    ("Loss decreases by >=10x on one batch",    float(overfit_loss_trace[0]) > 10.0 * float(overfit_loss_trace[-1])),
    ("Final loss strictly below initial",       float(overfit_loss_trace[-1]) < float(overfit_loss_trace[0])),
    ("Loss trace all finite",                   bool(np.isfinite(overfit_loss_trace).all())),
]
for description, condition in checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 13: Overfit Probe

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Loss plateaus far above 0 | model capacity too small, or a detached/blocked gradient path | Inspect parameter updates (Stage 6) and head width (Stage 2) |
| Loss never moves | non-deterministic dataloader yields a *different* batch each step | Confirm the repeated batch is identical across steps |
| Loss diverges | LR too high for a single-batch regime | Lower probe LR; watch for clip warnings |
| "Overfits" but real training fails | data augmentation / shuffling differs from the probe | Match the probe transform to the production loader |
| Probe never stops | `stop_threshold` below achievable float precision | Compare final loss to `stop_threshold` and `max_steps` |

**Passing to the summary:** every per-epoch sub-step and the capacity probe are individually verified; the final cell assembles the contract.

---
## End-to-End Summary

The cell below assembles a single table of every stage's inspected output — shapes,
types, and key statistics — and re-asserts the cross-stage contract. To run the *real*
`TrainingPipeline.run()` end-to-end once data is mounted, replace the synthetic batch
construction with:

```python
from configuration.training_config import TrainerConfig
from configuration.dataset_config  import DatasetConfiguration
from pipelines.training_pipeline.pipeline import TrainingPipeline

training_pipeline = TrainingPipeline(
    trainer_config = TRAINER_CONFIG,
    dataset_config = DatasetConfiguration(),
    model_name     = "unet",
    seed           = RANDOM_SEED,
)
train_losses, val_losses, best_val_loss = training_pipeline.run()
```

The assertions in each stage above then bracket any refactor of the training pipeline:
run this notebook before a change to establish the baseline, apply the change, and run
it again — the PASS/FAIL lines either still pass or pinpoint the regression.

In [ ]:
summary_rows = [
    ("1  Run metadata / seeding", "float32[N] x_axis",        f"N={SPECTRAL_X_AXIS.shape[0]}, span=[{SPECTRAL_X_AXIS.min():.1f},{SPECTRAL_X_AXIS.max():.1f}]"),
    ("2  Model + optimizer",      "nn.Module + AdamW",        f"out={tuple(model_forward_output.shape)}, params={trainable_parameter_count:,}, groups={len(adamw_optimizer.param_groups)}"),
    ("3  Warmup",                 "float ramp [α0,1]",         f"start={float(warmup_factor_trace[0]):.3f} -> end={float(warmup_factor_trace[-1]):.3f} over {warmup_step_count} steps"),
    ("4  LR scheduler",           "list[float] per group",     f"lr0={float(learning_rate_trace[0]):.2e} -> {float(learning_rate_trace[-1]):.2e}"),
    ("5  Loss",                   "dict total/comp/weighted",  f"total={float(loss_output_dictionary['total_loss']):.4f}, terms={list(loss_output_dictionary['components'].keys())}"),
    ("6  Train step",             "(loss, loss_dict)",         f"loss {float(repeated_step_losses[0]):.3e} -> {float(repeated_step_losses[-1]):.3e}"),
    ("7  Gradient clipper",       "clipped global norm",       f"max post-clip={float(post_clip_norms.max()):.3f} <= g_max={TRAINER_CONFIG.gradient_clipper.max_grad_norm}"),
    ("8  EMA",                    "shadow params dict",        f"divergence={float(divergence_trace[-1]):.4f}, restore err={maximum_restore_error:.2e}"),
    ("9  Metric aggregator",      "epoch-mean dicts",          f"count={metric_aggregator.count}, comps={list(reduced_components.keys())}"),
    ("10 Curriculum swap",        "bool + cfg replace",        f"swap@{curriculum_configuration.swap_epoch}, terms {active_terms_before_swap}->{active_terms_after_swap}"),
    ("11 Early stopping",         "bool stop flag",            f"plateau stop @ eval {first_stop_epoch}, best={improving_early_stop.best_loss:.4f}"),
    ("12 Checkpoint",             "best_model.pt dict",        f"keys={len(saved_checkpoint)}, resume global_step={global_step_after}"),
    ("13 Overfit probe",          "loss -> ~0",                f"{float(overfit_loss_trace[0]):.3e} -> {float(overfit_loss_trace[-1]):.3e}"),
]

header = f"{'Stage':<28}{'Output type':<28}{'Key statistics'}"
print(header)
print("-" * len(header))
for stage_name, output_type, statistics in summary_rows:
    print(f"{stage_name:<28}{output_type:<28}{statistics}")

In [ ]:
final_contract_checks = [
    ("Stage 1 -> 2: out_channels = 3K",          resolved_output_channels == OUTPUT_CHANNELS),
    ("Stage 2 -> 4: LR list len = num groups",   len(learning_rate_scheduler.step(0)) == len(adamw_optimizer.param_groups)),
    ("Stage 5 curve shape = (B, N, H, W)",       tuple(reconstructed_curves.shape) == (BATCH_SIZE, SPECTRAL_SAMPLE_POINTS, PATCH_HEIGHT, PATCH_WIDTH)),
    ("Stage 6 train step reduces loss",          float(repeated_step_losses[-1]) < float(repeated_step_losses[0])),
    ("Stage 7 clipping respects threshold",      bool(np.all(post_clip_norms <= TRAINER_CONFIG.gradient_clipper.max_grad_norm + 1e-4))),
    ("Stage 8 EMA restore is exact",             maximum_restore_error < 1e-6),
    ("Stage 11 early stopping triggers",         plateau_early_stop.triggered),
    ("Stage 12 checkpoint round-trips state",    global_step_after == global_step_before and scheduler_lrs_after == scheduler_lrs_before),
    ("Stage 13 overfit drives loss down",        float(overfit_loss_trace[0]) > 10.0 * float(overfit_loss_trace[-1])),
]
passed = sum(1 for _, condition in final_contract_checks if condition)
for description, condition in final_contract_checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")
print()
print(f"Cross-stage contract: {passed}/{len(final_contract_checks)} checks passed.")